# Dual Active Bridge (DAB) Converter — Transformer Design and Analysis

### Design Specifications

| Property | Symbol | Value | Unit |
|---|---|---|---|
| Nominal output power | $P_\mathrm{out,nom}$ | 50 | W |
| Nominal input voltage | $U_\mathrm{in,nom}$ | 50 | V |
| Minimum input voltage | $U_\mathrm{in,min}$ | 30 | V |
| Maximum input voltage | $U_\mathrm{in,max}$ | 60 | V |
| Output voltage | $U_\mathrm{out}$ | 15 | V |
| Output voltage ripple (rel.) | $\Delta U_\mathrm{out,pp,rel}$ | 2 | % |
| Switching frequency | $f_\mathrm{sw}$ | 40 | kHz |
| Auxiliary output voltage | $V_\mathrm{aux,out} $ | 18 | V |
| Auxiliary output power | $P_\mathrm{aux,out} $ | 5 | W |

In [7]:

import numpy as np
import matplotlib.pyplot as plt

In [8]:
# Configure matplotlib for high-resolution plots
plt.rcParams['figure.dpi'] = 150           # High-res display
plt.rcParams['savefig.dpi'] = 300          # High-res when saving
plt.rcParams['font.size'] = 10             # Readable font size
plt.rcParams['axes.linewidth'] = 1.2       # Slightly thicker axes
plt.rcParams['grid.alpha'] = 0.3           # Subtle grid
plt.rcParams['lines.linewidth'] = 2        # Thicker plot lines

In [9]:
# === Design Specifications ===
P_out_nom = 50          # Nominal output power [W]
U_in_nom  = 50          # Nominal input voltage [V]
U_in_min  = 30          # Minimum input voltage [V]
U_in_max  = 60          # Maximum input voltage [V]
U_out     = 12          # Output voltage [V]
dU_out_rel = 0.02       # Output voltage ripple (relative) [-]
f_sw      = 80e3        # Switching frequency [Hz]
V_aux_out = 18          # Auxiliary output voltage [V]
P_aux_out = 5           # Auxiliary output power [W]

# === Derived Quantities ===
T_sw      = 1 / f_sw                       # Switching period [s]
dU_out_pp = dU_out_rel * U_out             # Absolute output voltage ripple [V]

print(f"Switching period  T_sw  = {T_sw*1e6:.1f} µs")
print(f"Output ripple     ΔU_pp = {dU_out_pp*1e3:.0f} mV")

# === Selected Design Variables ===
N = 4.17                 # Transformer turns ratio [-]
C_in = 700e-6            # Input capacitor [F]
L = 26.04e-6             # Inductance [H]
C_DCblock = 15.20e-6     # DC-blocking capacitor [F]
C_out = 152.72e-6        # Output capacitor [F]

print(f"N          = {N:.2f} [-]")
print(f"C_in       = {C_in*1e6:.2f} µF")
print(f"L          = {L*1e6:.2f} µH")
print(f"C_DCblock  = {C_DCblock*1e6:.2f} µF")
print(f"C_out      = {C_out*1e6:.2f} µF")


Switching period  T_sw  = 12.5 µs
Output ripple     ΔU_pp = 240 mV
N          = 4.17 [-]
C_in       = 700.00 µF
L          = 26.04 µH
C_DCblock  = 15.20 µF
C_out      = 152.72 µF


## Q1: Calculation of Volt-Second Stress

The volt-second stress (peak flux-linkage) on the primary side is defined as the integral of the voltage applied to the transformer over one positive half cycle. In a DAB converter, both bridges operate at 50% duty cycle, so the full input voltage is applied for half the switching period:

$$\lambda_1 = U_\mathrm{in} \cdot \frac{T_\mathrm{sw}}{2}$$

The worst case (highest $\lambda_1$) occurs at **maximum input voltage** $U_\mathrm{in,max}$.

In [10]:
# Q1: Volt-second stress (primary side)
lambda_1_max = U_in_max * T_sw / 2

print(f"Worst-case volt-second stress: λ₁,max = {lambda_1_max*1e6:.1f} µV·s , ([V·s] = [Wb])")

Worst-case volt-second stress: λ₁,max = 375.0 µV·s , ([V·s] = [Wb])


## Q2: Estimation of Loss Coefficient $K_\mathrm{fe}$

The volumetric core loss at 80°C is given by the Steinmetz equation:

$$p_\mathrm{vol} = K_\mathrm{b} \cdot \left(\frac{f}{f_\mathrm{base}}\right)^{\alpha} \cdot \left(\frac{B_\mathrm{pk}}{B_\mathrm{base}}\right)^{\beta}$$

To express this as $p_\mathrm{vol} = K_\mathrm{fe} \cdot B_\mathrm{pk}^{\beta}$, we absorb the frequency dependence into $K_\mathrm{fe}$:

$$K_\mathrm{fe} = K_\mathrm{b} \cdot \left(\frac{f_\mathrm{sw}}{f_\mathrm{base}}\right)^{\alpha} \cdot \frac{1}{B_\mathrm{base}^{\beta}}$$

The nearest base frequency to $f_\mathrm{sw} = 80\,\mathrm{kHz}$ is **100 kHz**.

| $f_\mathrm{base}$ (kHz) | $\alpha$ | $\beta$ | $K_\mathrm{b}$ (kW/m³) | $B_\mathrm{base}$ (mT) |
|---|---|---|---|---|
| **100** | **1.72** | **2.68** | **61.15** | **100** |

In [11]:
# Q2: Core loss coefficient K_fe
# Steinmetz parameters for nearest base frequency (100 kHz row, N87 material)
f_base = 100e3          # Base frequency [Hz]
alpha  = 1.72           # Steinmetz frequency exponent [-]
beta   = 2.68           # Steinmetz flux exponent [-]
K_b    = 61.15e3        # Base volumetric loss coefficient [W/m³]
B_base = 100e-3         # Base peak flux density [T]

# K_fe such that p_vol = K_fe * B_pk^beta
K_fe = K_b * (f_sw / f_base)**alpha / B_base**beta

print(f"K_fe = {K_fe:.2f} W/(m³·T^{beta})")
print(f"K_fe = {K_fe/1e6:.4f} MW/(m³·T^{beta})")

K_fe = 19939361.12 W/(m³·T^2.68)
K_fe = 19.9394 MW/(m³·T^2.68)


## Q3: Total Current Stress

The total RMS current stress for the $K_g$ method is:

$$I_\mathrm{tot} = \sum_{j=1}^{k} n_j \cdot I_{j,\mathrm{RMS}}, \qquad n_j = \frac{N_j}{N_1}$$

For a DAB, the inductor current $i_L$ flows through the primary, and by transformer action $I_2 = N \cdot I_1$. Since $n_2 = 1/N$:

$$n_2 \cdot I_{2,\mathrm{RMS}} = \frac{1}{N} \cdot N \cdot I_{1,\mathrm{RMS}} = I_{1,\mathrm{RMS}}$$

Therefore:

$$I_\mathrm{tot} = 2 \cdot I_{1,\mathrm{RMS}} + n_\mathrm{aux} \cdot I_\mathrm{w,aux,RMS}$$

The worst case (highest RMS current) occurs at **minimum input voltage** $U_\mathrm{in,min}$, where the voltage mismatch causes higher circulating currents.

In [12]:
# Q3: Total current stress I_tot

# Given
I_aux_RMS = 0.8                          # Worst-case auxiliary RMS current [A]
n_aux = V_aux_out / U_in_nom             # Auxiliary turns ratio N_aux/N_1 ≈ V_aux/U_in_nom

# DAB worst case: U_in_min with nominal output power
V_1  = U_in_min                          # Primary-side voltage [V]
V_2p = N * U_out                         # Reflected secondary voltage to primary [V]
omega_L = 2 * np.pi * f_sw * L           # ωL [Ω]

# Phase shift δ from power equation:
#   P = V_1 · V_2' · δ · (π − δ) / (2π² · f_sw · L)
rhs = P_out_nom * 2 * np.pi**2 * f_sw * L / (V_1 * V_2p)
delta = (np.pi - np.sqrt(np.pi**2 - 4 * rhs)) / 2   # [rad]

# Current at switching instants (half-wave symmetry: i(π) = −i(0))
i_0     = -(V_1 * np.pi - V_2p * (np.pi - 2*delta)) / (2 * omega_L)
i_delta =  i_0 + (V_1 + V_2p) * delta / omega_L

# RMS current over half period [0, π] — piecewise-linear waveform
# Segment 1: [0, δ]  i(θ) = i_0 + slope_1·θ,  from i_0 → i_δ
slope_1 = (i_delta - i_0) / delta
int_1   = i_0**2 * delta + i_0 * slope_1 * delta**2 + slope_1**2 * delta**3 / 3

# Segment 2: [δ, π]  i(θ) = i_δ + slope_2·(θ−δ),  from i_δ → −i_0
dur_2   = np.pi - delta
slope_2 = (-i_0 - i_delta) / dur_2
int_2   = i_delta**2 * dur_2 + i_delta * slope_2 * dur_2**2 + slope_2**2 * dur_2**3 / 3

I_1_RMS = np.sqrt((int_1 + int_2) / np.pi)

# Total current stress
I_tot = 2 * I_1_RMS + n_aux * I_aux_RMS

print(f"Reflected secondary voltage  V₂' = N·U_out = {V_2p:.2f} V")
print(f"Phase shift                  δ   = {np.degrees(delta):.1f}°")
print(f"Current at θ=0               i₀  = {i_0:.3f} A")
print(f"Current at θ=δ               iδ  = {i_delta:.3f} A")
print(f"Primary RMS current          I₁,RMS = {I_1_RMS:.3f} A")
print(f"Auxiliary turns ratio         n_aux  = {n_aux:.3f}")
print(f"")
print(f"I_tot = 2 × {I_1_RMS:.3f} + {n_aux:.3f} × {I_aux_RMS:.1f} = {I_tot:.3f} A")

Reflected secondary voltage  V₂' = N·U_out = 50.04 V
Phase shift                  δ   = 30.0°
Current at θ=0               i₀  = 0.405 A
Current at θ=δ               iδ  = 3.604 A
Primary RMS current          I₁,RMS = 2.015 A
Auxiliary turns ratio         n_aux  = 0.360

I_tot = 2 × 2.015 + 0.360 × 0.8 = 4.317 A


## Q4: Window Utilization Factor

The window utilization factor $K_u$ (fill factor) accounts for insulation, bobbin walls, air gaps between round conductors, and the space lost to multiple winding layers.

For a bobbin-wound transformer with multiple windings and insulation layers, typical values are $K_u \approx 0.25 - 0.4$.

A conservative first estimate for our design: **$K_u = 0.3$**.

In [13]:
# Q4: Window utilization factor
K_u = 0.3   # Conservative estimate for bobbin-wound, multi-winding transformer [-]

print(f"Window utilization factor  K_u = {K_u}")

Window utilization factor  K_u = 0.3


## Q5: Estimation of Loss Budget

Target a reasonable converter efficiency $\eta$ and compute the total allowed losses:

$$P_\mathrm{tot} = P_\mathrm{out,nom} \cdot \frac{1 - \eta}{\eta}$$

For a 50 W DAB converter at 80 kHz (student design), a target efficiency of **$\eta = 95\%$** is a reasonable first estimate.

In [14]:
# Q5: Loss budget
eta = 0.95                                # Target converter efficiency [-]
P_tot = P_out_nom * (1 - eta) / eta       # Total allowed losses [W]

print(f"Target efficiency  η     = {eta*100:.0f} %")
print(f"Total loss budget  P_tot = {P_tot:.2f} W")

Target efficiency  η     = 95 %
Total loss budget  P_tot = 2.63 W
